# 4 Differential abundance analysis

## 1 Setup

### Load libraries

In [ ]:
library(readxl)
library(tidyverse)
library(Rvolcano)

### Assign workspace

In [ ]:
# Set working directory to project folder
wd <- "/path/to/02_metabolomics"
setwd(wd)

### Define functions
Helper functions for data cleaning, scaling, statistical tests, and feature filtering.

In [ ]:
# Replace zeros and NAs with small value (1/10 of smallest non-zero)
ReplaceZeroWithMin <- function(x) {
  x[x == 0 | is.na(x)] <- min(x[x > 0], na.rm = TRUE) / 10
  return(x)
}

# Replace NAs and falsy values with 0
ReplaceNaWithZero <- function(x) {
  x[!x | is.na(x)] <- 0
  return(x)
}

# Pareto scaling (mean-centered, divided by sqrt(sd))
paretoscale <- function(z) {
  rowmean <- apply(z, 1, mean, na.rm = TRUE)
  rowsd <- apply(z, 1, sd, na.rm = TRUE)
  rowsqrtsd <- sqrt(rowsd)
  rv <- sweep(z, 1, rowmean, "-")
  rv <- sweep(rv, 1, rowsqrtsd, "/")
  return(rv)
}

# Perform t-test between two groups of columns for one row
t_test <- function(row, group1_cols, group2_cols) {
  group1_values <- row[group1_cols]
  group2_values <- row[group2_cols]
  t_test_result <- t.test(group1_values, group2_values)
  return(t_test_result$p.value)
}

# Apply weightedMean; fall back to mean if no variability (MAD = 0)
safeWeightedMean <- function(x) {
  if (mad(x, na.rm = TRUE) == 0) {
    return(mean(x, na.rm = TRUE))
  } else {
    return(weightedMean(x))
  }
}

# Helper to filter features by attributes and return feature names
filter_features <- function(df, ...) {
  df %>%
    filter(...) %>%
    select(feature) %>%
    pull()
}

### Define parameters

In [ ]:
pvalueTH         <- 0.01      # Significance threshold for statistical tests
log2fcTH         <- 2         # Absolute log2 fold change threshold for relevant features
samplePercentTH  <- 0.8       # Required presence in % of samples per group (e.g. 80%)
absentPercentTH  <- 0.15      # Max allowed presence in 'absent' group (e.g. 15%)
retentiontimeTH  <- 0.1       # Minimum retention time in minutes (to filter noise)
intTH            <- 100000    # Minimum intensity threshold for relevant features

## 2 Load data

### Load Metadata

In [ ]:
# Load sample metadata
md_path <- "03_data/tables/samplelist.xlsx"
md.load <- read_excel(md_path)
md <- md.load %>% filter(type %in% c("sample", "control"))

# Load group definitions
mdgroups_path <- "03_data/tables/grouplist.xlsx"
mdgroups <- read_excel(mdgroups_path)

### Load and prepare data

In [ ]:
# Define ionization modes to process: positive and negative
modes <- c("pos", "neg")  

# Extract sample names from metadata
sampleType_list <- md %>% pull(name)

# Define key feature columns
featureCols <- c("feature", "id", "mode", "mz", "mz_range.min", "mz_range.max", 
                 "rt", "rt_range.min", "rt_range.max")

# Initialize lists to store data
fts_load <- list()
fts <- list()

# Iterate over positive and negative modes
for (mode_selected in modes) {
    
    # Construct file path to MZmine export
    ft_path <- file.path(wd, "04_analysis", "mzmine", mode_selected, "export_quant_modular.csv")
    
    # Load CSV
    ft_load <- read.csv(ft_path, sep = ",", header = TRUE, check.names = TRUE, row.names = 1)
    
    # Identify area columns
    colnames_area <- grep("^datafile\\..*\\.area$", names(ft_load), value = TRUE)
    
    # Subset and rename area + feature columns
    ft <- ft_load[, c("mz", "mz_range.min", "mz_range.max", "rt", "rt_range.min", 
                      "rt_range.max", colnames_area)]
    colnames(ft) <- gsub("\\.mzML\\.area|^datafile\\.", "", colnames(ft))
    
    # Add mode and id as explicit columns
    ft <- ft %>% mutate(mode = mode_selected, .before = 1)
    ft <- tibble::rownames_to_column(ft, var = "id")
    
    # Create unified feature name with padded ID
    max_length <- max(nchar(as.character(ft$id)))
    ft <- ft %>%
        mutate(feature = paste0(mode, "_", str_pad(id, max_length, pad = "0")), .before = 1) %>%
        select(all_of(c(featureCols, sampleType_list)))
    
    # Store in lists
    fts_load[[mode_selected]] <- ft_load
    fts[[mode_selected]] <- ft
}

# Merge positive and negative mode data frames
ft <- bind_rows(fts)

## 3 Statistical analysis





### Transform dataframes
Handle missing values, replace zeros, and create binary presence/absence matrices.


In [ ]:
# Get selected sample names and columns
md_selected_all.list <- md %>% pull(name)
md_selected_all.list.feature <- c("feature", md_selected_all.list)

# Select raw quantification table
ft_quant <- ft[md_selected_all.list.feature]

# Prepare table with mode and features for NA handling
md_selected_all.list.feature <- c("mode", "feature", md_selected_all.list)
ft_quant_noNA <- ft[md_selected_all.list.feature]
rownames(ft_quant_noNA) <- ft_quant_noNA$feature
ft_quant_noNA$feature <- NULL

# Positive mode
ft_quant_noNA_pos <- ft_quant_noNA %>%
  filter(mode == "pos") %>%
  dplyr::select(all_of(md_selected_all.list)) %>%
  apply(2, ReplaceNaWithZero) %>%
  as.data.frame()

ft_quant_noZero_pos <- ft_quant_noNA %>%
  filter(mode == "pos") %>%
  dplyr::select(all_of(md_selected_all.list)) %>%
  apply(2, ReplaceNaWithZero) %>%
  apply(2, ReplaceZeroWithMin) %>%
  as.data.frame()

# Negative mode
ft_quant_noNA_neg <- ft_quant_noNA %>%
  filter(mode == "neg") %>%
  dplyr::select(all_of(md_selected_all.list)) %>%
  apply(2, ReplaceNaWithZero) %>%
  as.data.frame()

ft_quant_noZero_neg <- ft_quant_noNA %>%
  filter(mode == "neg") %>%
  dplyr::select(all_of(md_selected_all.list)) %>%
  apply(2, ReplaceNaWithZero) %>%
  apply(2, ReplaceZeroWithMin) %>%
  as.data.frame()

# Combine positive and negative tables
ft_quant_noNA <- rbind(ft_quant_noNA_pos, ft_quant_noNA_neg)
ft_quant_noZero <- rbind(ft_quant_noZero_pos, ft_quant_noZero_neg)


### Calculate differential abundance across groups
Compute fold changes, weighted means, p-values, and sample counts between groups.


In [ ]:
# Initialize result table with feature column
ft_calc <- ft %>% dplyr::select(feature)

# Loop over all group comparisons
for (i in 1:nrow(mdgroups)) {
  
  # Extract group metadata
  groupName      <- mdgroups[i, ] %>% pull(groupName)
  ref_strain     <- mdgroups[i, ] %>% pull(refName)
  tst_strain     <- mdgroups[i, ] %>% pull(testName)
  ref_genotype   <- mdgroups[i, ] %>% pull(refGenotype)
  tst_genotype   <- mdgroups[i, ] %>% pull(testGenotype)
  
  md_selected.ref <- md %>% filter(strain_group == ref_strain)
  md_selected.tst <- md %>% filter(strain_group == tst_strain)

  # Print group and sample info
  cat("\n--- Group Information (Iteration", i, ") ---\n")
  cat(sprintf("%-20s %s\n", "Group Name:", groupName))
  cat(sprintf("%-20s %s\n", "Reference Strain:", ref_strain))
  cat(sprintf("%-20s %s\n", "Test Strain:", tst_strain))
  cat(sprintf("%-20s %s\n", "Reference Genotype:", ref_genotype))
  cat(sprintf("%-20s %s\n", "Test Genotype:", tst_genotype))
  cat("\n--- Sample Counts ---\n")
  cat(sprintf("%-35s %d\n", "Number of samples in Reference Group:", nrow(md_selected.ref)))
  cat(sprintf("%-35s %d\n", "Number of samples in Test Group:", nrow(md_selected.tst)))
  cat("\n------------------------------\n")
  flush.console()

  # Calculate kernel-weighted means on non-imputed data
  cat("Calculating kernel-weighted means for reference and test groups...\n")
  flush.console()
  ref_mean <- apply(ft_quant_noNA[md_selected.ref$name], 1, function(x) safeWeightedMean(x))
  tst_mean <- apply(ft_quant_noNA[md_selected.tst$name], 1, function(x) safeWeightedMean(x))
  
  # Compute log2 fold change
  ft_calc[[paste0(groupName, "_log2FC")]] <- log(tst_mean / ref_mean, 2)
  
  # Save group mean values
  cat("Calculating log2 fold change...\n")
  flush.console()
  ft_calc[[paste0(groupName, "_", ref_genotype, "_mean")]] <- ref_mean
  ft_calc[[paste0(groupName, "_", tst_genotype, "_mean")]] <- tst_mean

  # Calculate number of detected peaks (nonzero values)
  cat("Calculating peak numbers...\n")
  flush.console()
  ref_n <- rowSums(ft_quant_noNA[md_selected.ref$name] != 0)
  tst_n <- rowSums(ft_quant_noNA[md_selected.tst$name] != 0)
  ft_calc[[paste0(groupName, "_", ref_genotype, "_npeaks")]] <- ref_n
  ft_calc[[paste0(groupName, "_", tst_genotype, "_npeaks")]] <- tst_n
  ft_calc[[paste0(groupName, "_npeaks")]] <- ref_n + tst_n

  # Calculate kernel-weighted means on imputed data
  cat("Calculating log2 fold change for imputed data...\n")
  flush.console()
  ref_mean <- apply(ft_quant_noZero[md_selected.ref$name], 1, function(x) safeWeightedMean(x))
  tst_mean <- apply(ft_quant_noZero[md_selected.tst$name], 1, function(x) safeWeightedMean(x))
  ft_calc[[paste0(groupName, "_log2FCimp")]] <- log(tst_mean / ref_mean, 2)

  # Replace log2FCimp with NaN if both group means are zero
  cat("Applying mutation conditions...\n")
  flush.console()
  ft_calc <- ft_calc %>%
    mutate(
      !!paste0(groupName, "_log2FCimp") := ifelse(
        !!sym(paste0(groupName, "_", ref_genotype, "_mean")) == 0 &
        !!sym(paste0(groupName, "_", tst_genotype, "_mean")) == 0,
        NaN,
        !!sym(paste0(groupName, "_log2FCimp"))
      )
    )

  # Calculate p-values on pareto-scaled imputed data
  cat("Calculating p-values...\n")
  flush.console()
  p_values <- apply(paretoscale(ft_quant_noZero), 1, function(row) {
    p.valcalc(row[md_selected.ref$name], row[md_selected.tst$name])
  })
  ft_calc[[paste0(groupName, "_pvalue")]] <- p_values

  # Replace p-values with NaN if both group means are zero
  cat("Applying mutation conditions to p-values...\n")
  flush.console()
  ft_calc <- ft_calc %>%
    mutate(
      !!paste0(groupName, "_pvalue") := ifelse(
        !!sym(paste0(groupName, "_", ref_genotype, "_mean")) == 0 &
        !!sym(paste0(groupName, "_", tst_genotype, "_mean")) == 0,
        NaN,
        !!sym(paste0(groupName, "_pvalue"))
      )
    )
  
  cat("\n--- End of Iteration", i, "---\n")
  flush.console()
}


### Identify features passing blank and media thresholds
Filter features based on signal strength relative to blank and media control samples.

In [ ]:
# Calculate maximum mean across all group means
ft_calc <- ft_calc %>%
  mutate(max_mean = pmap_dbl(dplyr::select(., contains("_mean")), max, na.rm = TRUE))

# Process blank controls
md_ctrl_blank <- unique(md %>%
  filter(type == "control", sample %in% c("blank", "pblank")) %>%
  pull(sample))

for (ctrl in md_ctrl_blank) {
  md_selected <- md %>% filter(sample == ctrl)
  ft_calc[[paste0("blank_max_", ctrl)]] <- apply(ft_quant_noNA[md_selected$name], 1, max, na.rm = TRUE)
}

ft_calc <- ft_calc %>%
  mutate(
    max_blank = pmap_dbl(dplyr::select(., contains("blank_max_")), max, na.rm = TRUE),
    blank_pass = max_mean >= 10 * max_blank
  )

# Process media controls
md_ctrl_media <- unique(md %>%
  filter(type == "control", sample %in% c("SM5", "KA")) %>%
  pull(sample))

for (ctrl in md_ctrl_media) {
  md_selected <- md %>% filter(sample == ctrl)
  ft_calc[[paste0("media_max_", ctrl)]] <- apply(ft_quant_noNA[md_selected$name], 1, max, na.rm = TRUE)
}

ft_calc <- ft_calc %>%
  mutate(
    max_media = pmap_dbl(dplyr::select(., contains("media_max_")), max, na.rm = TRUE),
    media_pass = max_mean >= 4 * max_media
  )

# Combine blank and media filters into final flag
ft_calc <- ft_calc %>%
  mutate(ctrl_pass = blank_pass & media_pass)

# Merge calculated features back with combined table
ft <- merge(ft, ft_calc)


### Adjusting p-values for each stage
Apply intensity and sample count thresholds to retain only robustly detected features before performing Benjamini-Hochberg (BH) p-value adjustment.

In [ ]:
ft <- ft %>%
  group_by(mode) %>%
  mutate(
    # Apply sample intensity and presence thresholds per stage
    veg_pass = (veg_wt_mean >= intTH | veg_ko_mean >= intTH) &
               (veg_wt_npeaks >= 18 * samplePercentTH & veg_ko_npeaks >= 18 * samplePercentTH),
    stv_pass = (stv_wt_mean >= intTH | stv_ko_mean >= intTH) &
               (stv_wt_npeaks >= 6 * samplePercentTH & stv_ko_npeaks >= 6 * samplePercentTH),
    fbs_pass = (fbs_wt_mean >= intTH | fbs_ko_mean >= intTH) &
               (fbs_wt_npeaks >= 11 * samplePercentTH & fbs_ko_npeaks >= 11 * samplePercentTH),

    # Adjust p-values with BH correction for passed features
    veg_pvalue_nonadj = ifelse(veg_pass, veg_pvalue, NA),
    veg_pvalue        = ifelse(veg_pass, p.adjust(veg_pvalue_nonadj, method = "BH"), NA),

    stv_pvalue_nonadj = ifelse(stv_pass, stv_pvalue, NA),
    stv_pvalue        = ifelse(stv_pass, p.adjust(stv_pvalue_nonadj, method = "BH"), NA),

    fbs_pvalue_nonadj = ifelse(fbs_pass, fbs_pvalue, NA),
    fbs_pvalue        = ifelse(fbs_pass, p.adjust(fbs_pvalue_nonadj, method = "BH"), NA)
  ) %>%
  ungroup()

## 4 Annotation





### Mark features with MS2 data
Flag features that have corresponding MS2 fragmentation spectra for downstream prioritization.


In [ ]:
# Load GNPS MS2 quant export files for positive and negative mode
ft_msms_path_pos <- file.path(wd, "04_analysis", "mzmine", "pos", "export_gnps_quant.csv")
ft_msms_pos <- read.csv(ft_msms_path_pos, header = TRUE, check.names = TRUE) %>% arrange(row.ID)

ft_msms_path_neg <- file.path(wd, "04_analysis", "mzmine", "neg", "export_gnps_quant.csv")
ft_msms_neg <- read.csv(ft_msms_path_neg, header = TRUE, check.names = TRUE) %>% arrange(row.ID)

# Assign mode and combine into one table
ft_msms_pos$mode <- "pos"
ft_msms_neg$mode <- "neg"

# Combine positive and negative mode tables and generate standardized feature IDs
ft_msms <- rbind(ft_msms_pos %>% select(row.ID, mode), ft_msms_neg %>% select(row.ID, mode)) %>%
  mutate(feature = paste0(mode, "_", str_pad(row.ID, max_length, pad = "0")), .before = 1)

# Create feature list and flag MS2 features in merged table
ft_msms_list <- ft_msms %>% pull(feature)
ft$msms <- ft$feature %in% ft_msms_list

### Add MZmine annotations
Integrate MZmine alignment, ion identity, lipid, and molecular networking annotations.


In [ ]:
# Define MZmine3 annotation columns to extract
colnames_extract <- c(
    'alignment_scores.rate', 'alignment_scores.aligned_features_n', 'alignment_scores.align_extra_features',
    'alignment_scores.weighted_distance_score', 'alignment_scores.mz_diff_ppm', 'alignment_scores.mz_diff',
    'alignment_scores.rt_absolute_error', 'alignment_scores.ion_mobility_absolute_error', 'feature_group',
    'ion_identities.iin_id', 'ion_identities.ion_identities', 'ion_identities.list_size', 'ion_identities.neutral_mass',
    'ion_identities.partner_row_ids', 'ion_identities.iin_relationship', 'ion_identities.consensus_formulas',
    'ion_identities.simple_formulas', 'lipid_annotations.lipid_annotations', 'lipid_annotations.ion_adduct',
    'lipid_annotations.mol_formula', 'lipid_annotations.mz_diff_ppm', 'lipid_annotations.explained_intensity_percent',
    'molecular_networking.net_cluster_id', 'molecular_networking.net_community_id'
)

modes <- c("pos", "neg")
fts_load_mzmine <- list()
fts_mzmine <- list()

# Process each mode and extract relevant annotations
for (mode_selected in modes) {
    ft_path <- file.path(wd, "04_analysis", "mzmine", mode_selected, "export_quant_modular.csv")
    ft_load_mzmine <- read.csv(ft_path, sep = ",", header = TRUE, check.names = TRUE, row.names = 1)

    ft_load_mzmine <- ft_load_mzmine[, colnames_extract]
    colnames(ft_load_mzmine) <- gsub('\\.mzML.area|^datafile\\.', '', colnames(ft_load_mzmine))
    ft_load_mzmine <- ft_load_mzmine %>% mutate(mode = mode_selected, .before = 1)
    ft_load_mzmine <- tibble::rownames_to_column(ft_load_mzmine, var = "id")

    max_length <- max(nchar(as.character(ft_load_mzmine$id)))
    ft_load_mzmine <- ft_load_mzmine %>%
        mutate(feature = paste0(mode, "_", str_pad(id, max_length, pad = "0")), .before = 1) %>%
        select(-id)

    fts_load_mzmine[[mode_selected]] <- ft_load_mzmine
    fts_mzmine[[mode_selected]] <- ft_load_mzmine
}

# Combine and merge with main feature table
ft_mzmine <- bind_rows(fts_mzmine)
ft <- merge(ft, ft_mzmine, all.x = TRUE)


### Add SIRIUS annotations

Integrates CANOPUS class predictions and compound identifications from SIRIUS into the feature table for downstream use.


In [ ]:
# Load and process SIRIUS CANOPUS annotations for positive mode
canopus_pos_path <- file.path(wd, "04_analysis", "sirius", "pos", "canopus_compound_summary.tsv")
canopus_pos <- read.csv(canopus_pos_path, sep = "\t", header = TRUE, check.names = TRUE) %>%
  select(-ClassyFire.all.classifications) %>%
  group_by(featureId) %>% arrange(desc(NPC.pathway.Probability)) %>% slice_head(n = 1) %>%
  ungroup() %>%
  mutate(mode = "pos")

canopus_pos <- canopus_pos %>%
  mutate(id = featureId, .before = 1) %>%
  mutate(id = sprintf(paste0("%0", max(nchar(as.character(id))), "d"), id)) %>%
  mutate(id = paste(mode, id, sep = "_"), .before = 3)

# Repeat for negative mode
canopus_neg_path <- file.path(wd, "04_analysis", "sirius", "neg", "canopus_compound_summary.tsv")
canopus_neg <- read.csv(canopus_neg_path, sep = "\t", header = TRUE, check.names = TRUE) %>%
  select(-ClassyFire.all.classifications) %>%
  group_by(featureId) %>% arrange(desc(NPC.pathway.Probability)) %>% slice_head(n = 1) %>%
  ungroup() %>%
  mutate(mode = "neg")

canopus_neg <- canopus_neg %>%
  mutate(id = featureId, .before = 1) %>%
  mutate(id = sprintf(paste0("%0", max(nchar(as.character(id))), "d"), id)) %>%
  mutate(id = paste(mode, id, sep = "_"), .before = 3)

# Load and process compound identifications (positive + negative)
ident_pos_path <- file.path(wd, "04_analysis", "sirius", "pos", "compound_identifications.tsv")
ident_pos <- read.csv(ident_pos_path, sep = "\t", header = TRUE, check.names = FALSE) %>%
  group_by(featureId) %>% slice_max(ConfidenceScore, with_ties = FALSE) %>%
  ungroup()
colnames(ident_pos) <- gsub("#", "", colnames(ident_pos))

ident_neg_path <- file.path(wd, "04_analysis", "sirius", "neg", "compound_identifications.tsv")
ident_neg <- read.csv(ident_neg_path, sep = "\t", header = TRUE, check.names = FALSE) %>%
  group_by(featureId) %>% slice_max(ConfidenceScore, with_ties = FALSE) %>%
  ungroup()
colnames(ident_neg) <- gsub("#", "", colnames(ident_neg))
ident_neg <- ident_neg %>% select(featureId, name, smiles)

# Merge CANOPUS + identifications, positive + negative
sirius_pos <- merge(canopus_pos, ident_pos, all.x = TRUE)
sirius_neg <- merge(canopus_neg, ident_neg, all.x = TRUE)
sirius_merge <- bind_rows(sirius_pos, sirius_neg)

# Add "sirius_" prefix to column names, fix feature column
colnames(sirius_merge) <- paste0("sirius_", colnames(sirius_merge))
names(sirius_merge)[names(sirius_merge) == "sirius_id"] <- "feature"
sirius_merge <- sirius_merge %>% select(-sirius_featureId)

# Merge into feature tables
ft_sirius <- ft[c("feature", "mode", "id", "mz", "rt")]
ft_sirius <- merge(ft_sirius, sirius_merge, all = TRUE)
ft <- merge(ft, ft_sirius, all.x = TRUE)

# Clean up NPC and ClassyFire columns: replace NA / null / empty with "unknown"
sirius_npc_columns <- grep("^sirius_NPC", colnames(ft), value = TRUE)
sirius_npc_columns <- sirius_npc_columns[!grepl("Probability", sirius_npc_columns)]

# Replace NA / null / empty with "unknown"
ft <- ft %>%
  mutate(across(all_of(sirius_npc_columns), ~if_else(. %in% c(NA, "null", ""), "unknown", .)))

# Clean up ClassyFire columns: replace NA / null / empty with "unknown"
sirius_classyfire_columns <- grep("sirius_ClassyFire", colnames(ft), value = TRUE)
sirius_classyfire_columns <- sirius_classyfire_columns[!grepl("Probability|probability", sirius_classyfire_columns)]

# Replace NA / null / empty with "unknown"
ft <- ft %>%
  mutate(across(all_of(sirius_classyfire_columns), ~if_else(. %in% c(NA, "null", ""), "unknown", .)))

# Clean up SIRIUS name column: replace NA / null / empty with "unknown"
ft <- ft %>%
  mutate(across(all_of("sirius_name"), ~if_else(. %in% c(NA, "null", ""), "unknown", .)))


## 5 Classification





### Differential abundance across stages

This classifies features as **absent**, **downregulated**, **upregulated**, or **unchanged**  
(`abs`, `down`, `up`, `none`) within each stage (`veg`, `stv`, `fbs`),  
based on abundance thresholds, fold change, p-values, and sample counts.

If none of the criteria are fulfilled, the feature is designated as **not assigned** (`na`) for that stage.

**Important:** For the vegetative stage (`veg`), a lower intensity threshold is applied.  
In general, this threshold is set lower than for differential abundance to also retain less abundant features.

In [ ]:
# Helper to filter features by dynamic stage
filter_stage_features <- function(stage, type) {
  prefix <- paste0(stage, "_")
  wt_mean <- paste0(prefix, "wt_mean")
  ko_mean <- paste0(prefix, "ko_mean")
  log2fc  <- paste0(prefix, "log2FC")
  pval    <- paste0(prefix, "pvalue")
  wt_npk  <- paste0(prefix, "wt_npeaks")
  ko_npk  <- paste0(prefix, "ko_npeaks")
  
  if (type == "absent") {
    filter_features(ft,
      !!sym(wt_mean) >= intTH / ifelse(stage == "veg", 100, 10) | msms == TRUE,
      !!sym(log2fc) <= 0,
      !!sym(wt_npk) >= c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * samplePercentTH,
      !!sym(ko_npk) <= ceiling(c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * absentPercentTH),
      !!sym(ko_mean) <= 100,
      rt >= retentiontimeTH,
      blank_pass == TRUE,
      media_pass == TRUE
    )
  } else if (type == "down") {
    filter_features(ft,
      !!sym(wt_mean) >= intTH | msms == TRUE,
      !!sym(log2fc) <= -log2fcTH,
      !!sym(pval) <= pvalueTH,
      !!sym(wt_npk) >= c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * samplePercentTH &
      !!sym(ko_npk) >= c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * samplePercentTH,
      rt >= retentiontimeTH,
      blank_pass == TRUE,
      media_pass == TRUE
    )
  } else if (type == "up") {
    filter_features(ft,
      !!sym(ko_mean) >= intTH | msms == TRUE,
      !!sym(log2fc) >= log2fcTH,
      !!sym(pval) <= pvalueTH,
      !!sym(wt_npk) >= c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * samplePercentTH &
      !!sym(ko_npk) >= c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * samplePercentTH,
      rt >= retentiontimeTH,
      blank_pass == TRUE,
      media_pass == TRUE
    )
  } else if (type == "none") {
    filter_features(ft,
      (!!sym(ko_mean) >= intTH | !!sym(wt_mean) >= intTH) | msms == TRUE,
      (!!sym(log2fc) > -log2fcTH | !!sym(pval) > pvalueTH),
      (!!sym(log2fc) < log2fcTH  | !!sym(pval) > pvalueTH),
      !!sym(wt_npk) >= c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * samplePercentTH |
      !!sym(ko_npk) >= c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * samplePercentTH,
      rt >= retentiontimeTH,
      blank_pass == TRUE,
      media_pass == TRUE
    )
  }
}

# Create a list with each stage
stages <- c("veg", "stv", "fbs")

# Ensure missing feature_* columns exist and are FALSE
for (stage in stages) {
  for (type in c("absent", "down", "up", "none")) {
    col_name <- paste0("feature_", type, "_", stage)
    if (!(col_name %in% colnames(ft))) {
      ft[[col_name]] <- FALSE
    }
  }
}

# Assign summary change labels per stage
for (stage in stages) {
  ft[[paste0("change_", stage)]] <- case_when(
    ft[[paste0("feature_absent_", stage)]] ~ "abs",
    ft[[paste0("feature_down_", stage)]] ~ "down",
    ft[[paste0("feature_up_", stage)]] ~ "up",
    ft[[paste0("feature_none_", stage)]] ~ "none",
    TRUE ~ "na"
  )
}

### Absence across stages
Summarizes absence patterns for vegetative, starved, and fruiting stages (`metaChange_abs`)

In [ ]:
# Consolidate all updates into a single mutate call for efficiency
ft <- ft %>%
  mutate(             
    # Update metaChange_abs based on absence conditions in both vegetative and starved
    metaChange_abs = case_when(
      feature_absent_veg == TRUE & feature_absent_stv == TRUE & feature_absent_fbs == TRUE ~ "veg-stv-fbs",
      feature_absent_veg == FALSE & feature_absent_stv == TRUE & feature_absent_fbs == TRUE ~ "stv-fbs",
      feature_absent_veg == TRUE & feature_absent_stv == TRUE & feature_absent_fbs == FALSE ~ "veg-stv",
      feature_absent_veg == TRUE & feature_absent_stv == FALSE & feature_absent_fbs == TRUE ~ "veg-fbs",
      feature_absent_veg == TRUE ~ "veg",
      feature_absent_stv == TRUE ~ "stv",
      feature_absent_fbs == TRUE ~ "fbs",
      TRUE ~ "na"
    ),
  )

### Absence across *pks* null mutants

Features with no detected peaks in PKS knockout mutants (`pks5`, `pks7`, `pks24`, `pks25`, `pks2425`),  
but present in all wild-type samples, are identified.  

Biologically meaningful absence patterns, including single and combined mutant profiles,  
are summarized in the `metaChange_pks` column.



In [ ]:
# Helper to filter features by dynamic stage
filter_stage_features <- function(stage, type) {
  prefix <- paste0(stage, "_")
  wt_mean <- paste0(prefix, "wt_mean")
  ko_mean <- paste0(prefix, "ko_mean")
  log2fc  <- paste0(prefix, "log2FC")
  pval    <- paste0(prefix, "pvalue")
  wt_npk  <- paste0(prefix, "wt_npeaks")
  ko_npk  <- paste0(prefix, "ko_npeaks")
  
  if (type == "absent") {
    filter_features(ft,
      !!sym(wt_mean) >= intTH / ifelse(stage == "veg", 100, 10) | msms == TRUE,
      !!sym(log2fc) <= 0,
      !!sym(wt_npk) >= c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * samplePercentTH,
      !!sym(ko_npk) <= ceiling(c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * absentPercentTH),
      !!sym(ko_mean) <= 100,
      rt >= retentiontimeTH,
      blank_pass == TRUE,
      media_pass == TRUE
    )
  } else if (type == "down") {
    filter_features(ft,
      !!sym(wt_mean) >= intTH | msms == TRUE,
      !!sym(log2fc) <= -log2fcTH,
      !!sym(pval) <= pvalueTH,
      !!sym(wt_npk) >= c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * samplePercentTH &
      !!sym(ko_npk) >= c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * samplePercentTH,
      rt >= retentiontimeTH,
      blank_pass == TRUE,
      media_pass == TRUE
    )
  } else if (type == "up") {
    filter_features(ft,
      !!sym(ko_mean) >= intTH | msms == TRUE,
      !!sym(log2fc) >= log2fcTH,
      !!sym(pval) <= pvalueTH,
      !!sym(wt_npk) >= c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * samplePercentTH &
      !!sym(ko_npk) >= c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * samplePercentTH,
      rt >= retentiontimeTH,
      blank_pass == TRUE,
      media_pass == TRUE
    )
  } else if (type == "none") {
    filter_features(ft,
      (!!sym(ko_mean) >= intTH | !!sym(wt_mean) >= intTH) | msms == TRUE,
      (!!sym(log2fc) > -log2fcTH | !!sym(pval) > pvalueTH),
      (!!sym(log2fc) < log2fcTH  | !!sym(pval) > pvalueTH),
      !!sym(wt_npk) >= c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * samplePercentTH |
      !!sym(ko_npk) >= c(18, 6, 11)[which(c("veg", "stv", "fbs") == stage)] * samplePercentTH,
      rt >= retentiontimeTH,
      blank_pass == TRUE,
      media_pass == TRUE
    )
  }
}

# Create a list with each stage
stages <- c("veg", "stv", "fbs")

# Ensure missing feature_* columns exist and are FALSE
for (stage in stages) {
  for (type in c("absent", "down", "up", "none")) {
    col_name <- paste0("feature_", type, "_", stage)
    if (!(col_name %in% colnames(ft))) {
      ft[[col_name]] <- FALSE
    }
  }
}

# Assign summary change labels per stage
for (stage in stages) {
  ft[[paste0("change_", stage)]] <- case_when(
    ft[[paste0("feature_absent_", stage)]] ~ "abs",
    ft[[paste0("feature_down_", stage)]] ~ "down",
    ft[[paste0("feature_up_", stage)]] ~ "up",
    ft[[paste0("feature_none_", stage)]] ~ "none",
    TRUE ~ "na"
  )
}

### Absence accross stages
Summarizes absence patterns for vegetative, starved, and fruiting stages (`metaChange_abs`)

In [ ]:
# Consolidate all updates into a single mutate call for efficiency
ft <- ft %>%
  mutate(             
    # Update metaChange_abs based on absence conditions in both vegetative and starved
    metaChange_abs = case_when(
      feature_absent_veg == TRUE & feature_absent_stv == TRUE & feature_absent_fbs == TRUE ~ "veg-stv-fbs",
      feature_absent_veg == FALSE & feature_absent_stv == TRUE & feature_absent_fbs == TRUE ~ "stv-fbs",
      feature_absent_veg == TRUE & feature_absent_stv == TRUE & feature_absent_fbs == FALSE ~ "veg-stv",
      feature_absent_veg == TRUE & feature_absent_stv == FALSE & feature_absent_fbs == TRUE ~ "veg-fbs",
      feature_absent_veg == TRUE ~ "veg",
      feature_absent_stv == TRUE ~ "stv",
      feature_absent_fbs == TRUE ~ "fbs",
      TRUE ~ "na"
    ),
  )

### Absence accross pks mutants in fruiting bodies
This section identifies features that show no detected peaks in the PKS knockout mutants (`pks5`, `pks7`, `pks24`, `pks25`, `pks2425`),  
while being present in all wild-type samples. It then summarizes biological meaningful absence patterns across the pks mutants including combinations (`metaChange_pks`)

In [ ]:
# Define PKS mutants to process
pks_mutants <- c("pks5", "pks7", "pks24", "pks25", "pks2425")

# Loop over PKS mutants
for (pks in pks_mutants) {
  log2fc_col <- paste0(pks, "_log2FC")
  wt_npk_col <- paste0(pks, "_wt_npeaks")
  ko_npk_col <- paste0(pks, "_ko_npeaks")
  
  feature_list <- filter_features(
    ft,
    !!sym(wt_npk_col) >= 3,
    !!sym(ko_npk_col) == 0,
    rt >= retentiontimeTH,
    blank_pass == TRUE,
    media_pass == TRUE
  )
  
  ft[[paste0("feature_absent_", pks)]] <- ft$feature %in% feature_list
}

# Consolidate PKS absence patterns into a single mutate call
ft <- ft %>%
  mutate(
    metaChange_pks = case_when(
      # Single mutant absence patterns
      feature_absent_fbs == TRUE & feature_absent_pks5 == TRUE  & feature_absent_pks7 == FALSE & feature_absent_pks24 == FALSE & feature_absent_pks25 == FALSE & feature_absent_pks2425 == FALSE ~ "pks5",
      feature_absent_fbs == TRUE & feature_absent_pks7 == TRUE  & feature_absent_pks5 == FALSE & feature_absent_pks24 == FALSE & feature_absent_pks25 == FALSE & feature_absent_pks2425 == FALSE ~ "pks7",
      
      # Combined mutant absence patterns
      feature_absent_fbs == TRUE & feature_absent_pks24 == TRUE & feature_absent_pks25 == TRUE  & feature_absent_pks2425 == TRUE  & feature_absent_pks5 == FALSE & feature_absent_pks7 == FALSE ~ "pks24_pks25_pks2425",
      feature_absent_fbs == TRUE & feature_absent_pks24 == TRUE & feature_absent_pks25 == TRUE  & feature_absent_pks2425 == FALSE & feature_absent_pks5 == FALSE & feature_absent_pks7 == FALSE ~ "pks24andpks25",
      feature_absent_fbs == TRUE & feature_absent_pks24 == TRUE & feature_absent_pks25 == FALSE & feature_absent_pks2425 == FALSE & feature_absent_pks5 == FALSE & feature_absent_pks7 == FALSE ~ "pks24",
      feature_absent_fbs == TRUE & feature_absent_pks25 == TRUE & feature_absent_pks24 == FALSE & feature_absent_pks2425 == FALSE & feature_absent_pks5 == FALSE & feature_absent_pks7 == FALSE ~ "pks25",
      feature_absent_fbs == TRUE & feature_absent_pks2425 == TRUE & feature_absent_pks24 == FALSE & feature_absent_pks25 == FALSE & feature_absent_pks5 == FALSE & feature_absent_pks7 == FALSE ~ "pks2425",

      # General PKS-related absence (any of the above absent)
      feature_absent_fbs == TRUE & (feature_absent_pks5 == TRUE | feature_absent_pks7 == TRUE | feature_absent_pks24 == TRUE | feature_absent_pks25 == TRUE | feature_absent_pks2425 == TRUE) ~ "pks-related",
      
      # General DiSfp absence pattern
      feature_absent_fbs == TRUE ~ "disfp",
      
      # Default: not assigned
      TRUE ~ "na"
    )
  )


## 6 Save outputs
Save the final processed feature table to a CSV file for downstream analysis and archiving.


In [ ]:
# Define the output file path for the final processed feature table
ft_processedAll_filePath <- file.path(wd, "04_analysis", "tables", "featureTable.csv")

# Create the output directory if it doesn’t exist
dir.create(dirname(ft_processedAll_filePath), recursive = TRUE)

# Write the processed feature table to CSV without row names
write.csv(ft, ft_processedAll_filePath, row.names = FALSE)